# Contrastive Learning for Next-Action Prediction

This notebook demonstrates the contrastive learning approach for predicting user next actions.

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import torch
from pathlib import Path

from pipeline.contrastive_model import SessionDataset, ContrastiveRecommender, ContrastiveTrainer
from scripts.run_contrastive import ContrastivePipeline


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Alejandro\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Alejandro\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Alejandro\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
  

## Initialize Pipeline

In [ ]:
data_path = Path('../Data')
pipeline = ContrastivePipeline(data_path=str(data_path))

print(f"Device: {pipeline.device}")

## Load and Explore Data

In [ ]:
x_train = pipeline.load_data()

print(f"\nDataset shape: {x_train.shape}")
print(f"\nColumns: {x_train.columns.tolist()}")
print(f"\nSample row:")
x_train.head(2)

In [ ]:
print(f"Total unique jobs: {len(pipeline.all_jobs)}")
print(f"Job ID range: {min(pipeline.all_jobs)} - {max(pipeline.all_jobs)}")

## Analyze Action Distribution

In [ ]:
import ast

all_actions = []
for actions in x_train['actions']:
    action_list = ast.literal_eval(actions) if isinstance(actions, str) else actions
    all_actions.extend(action_list)

from collections import Counter
action_counts = Counter(all_actions)

print("Action distribution:")
for action, count in action_counts.items():
    print(f"{action}: {count} ({count/len(all_actions)*100:.2f}%)")

## Create Dataloaders

In [ ]:
train_loader, val_loader = pipeline.prepare_dataloaders(
    x_train, batch_size=64, val_split=0.1
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## Inspect Sample Batch

In [ ]:
sample_batch = next(iter(train_loader))

print("Batch contents:")
for key, value in sample_batch.items():
    if isinstance(value, torch.Tensor):
        print(f"{key}: shape {value.shape}, dtype {value.dtype}")
    else:
        print(f"{key}: {type(value)}")

## Build Model

In [ ]:
pipeline.build_model(num_jobs=max(pipeline.all_jobs) + 100)

print("\nModel architecture:")
print(pipeline.model)

## Train Model

In [ ]:
history = pipeline.train(
    train_loader,
    num_epochs=10,
    lr=0.001
)

## Visualize Training

In [ ]:
import matplotlib.pyplot as plt

epochs = range(len(history))
losses = [h['loss'] for h in history]
pos_sims = [h['pos_similarity'] for h in history]
neg_sims = [h['neg_similarity'] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(epochs, losses, marker='o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True)

axes[1].plot(epochs, pos_sims, marker='o', label='Positive')
axes[1].plot(epochs, neg_sims, marker='s', label='Negative')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Similarity')
axes[1].set_title('Positive vs Negative Similarity')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Test Prediction

In [ ]:
test_sample = x_train.iloc[:5]

predictions = pipeline.predict(test_sample, top_k=10)

for i, pred in enumerate(predictions):
    print(f"\nSession {i}:")
    print(f"Predicted jobs: {pred}")
    print(f"Actual sequence: {test_sample.iloc[i]['job_ids']}")

## Save Model

In [ ]:
pipeline.trainer.save_model('../experiments/contrastive_model.pt')

## Embedding Visualization

In [ ]:
from sklearn.manifold import TSNE

pipeline.model.eval()

sample_sessions = x_train.sample(200)
sample_dataset = SessionDataset(sample_sessions)
sample_loader = torch.utils.data.DataLoader(sample_dataset, batch_size=64, shuffle=False)

embeddings = []
labels = []

with torch.no_grad():
    for batch in sample_loader:
        job_seq = batch['job_seq'].to(pipeline.device)
        action_seq = batch['action_seq'].to(pipeline.device)
        features = batch['features'].to(pipeline.device)
        
        emb = pipeline.model.get_session_embedding(job_seq, action_seq, features)
        embeddings.append(emb.cpu().numpy())
        
        has_apply = (action_seq.sum(dim=1) > 0).cpu().numpy()
        labels.append(has_apply)

embeddings = np.vstack(embeddings)
labels = np.hstack(labels)

print(f"Embedding shape: {embeddings.shape}")
print(f"Sessions with apply: {labels.sum()}/{len(labels)}")

In [ ]:
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embeddings_2d = tsne.fit_transform(embeddings)

plt.figure(figsize=(10, 8))
plt.scatter(embeddings_2d[labels==0, 0], embeddings_2d[labels==0, 1], 
           alpha=0.6, label='View only', s=30)
plt.scatter(embeddings_2d[labels==1, 0], embeddings_2d[labels==1, 1], 
           alpha=0.6, label='Has apply', s=30)
plt.xlabel('TSNE 1')
plt.ylabel('TSNE 2')
plt.title('Session Embeddings (TSNE projection)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()